In [1]:
!pip install pymupdf sentence-transformers faiss-cpu streamlit google-generativeai pyngrok faiss-cpu

In [2]:
import urllib.request
import os

papers = {
    "attention.pdf": "https://arxiv.org/pdf/1706.03762.pdf",
    "bert.pdf": "https://arxiv.org/pdf/1810.04805.pdf",
    "gpt3.pdf": "https://arxiv.org/pdf/2005.14165.pdf",
    "rag.pdf": "https://arxiv.org/pdf/2005.11401.pdf",
    "sbert.pdf": "https://arxiv.org/pdf/1908.10084.pdf",
    "lora.pdf": "https://arxiv.org/pdf/2106.09685.pdf",
    "llama2.pdf": "https://arxiv.org/pdf/2307.09288.pdf"
}

os.makedirs("papers", exist_ok=True)

for filename, url in papers.items():
    filepath = f"papers/{filename}"
    if not os.path.exists(filepath):
        print(f"downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"done")
    else:
        print(f"{filename} already exists")
#just downloading all pdfs and making a nice directory to store them to access later
print("all papers ready")

attention.pdf already exists
bert.pdf already exists
gpt3.pdf already exists
rag.pdf already exists
sbert.pdf already exists
lora.pdf already exists
llama2.pdf already exists
all papers ready


In [3]:
import fitz  # pymupdf

def extract_text_from_pdf(filepath):
    doc = fitz.open(filepath)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

# extract text from all papers
paper_texts = {}
for filename in papers.keys():
    filepath = f"papers/{filename}"
    text = extract_text_from_pdf(filepath)
    paper_texts[filename] = text
    print(f"{filename}: {len(text)} characters extracted")

attention.pdf: 39498 characters extracted
bert.pdf: 64117 characters extracted
gpt3.pdf: 236670 characters extracted
rag.pdf: 69078 characters extracted
sbert.pdf: 44198 characters extracted
lora.pdf: 82694 characters extracted
llama2.pdf: 263812 characters extracted


In [4]:
def split_into_chunks(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []

    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap

    return chunks

# chunk all papers, each into size of 500 and a overlap of 50 between each chunk to give clear continuation to the llm
all_chunks = []
chunk_sources = []

for filename, text in paper_texts.items():
    chunks = split_into_chunks(text)
    all_chunks.extend(chunks)
    chunk_sources.extend([filename] * len(chunks))
    print(f"{filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print(chunk_sources)

attention.pdf: 14 chunks
bert.pdf: 23 chunks
gpt3.pdf: 85 chunks
rag.pdf: 22 chunks
sbert.pdf: 15 chunks
lora.pdf: 31 chunks
llama2.pdf: 95 chunks

Total chunks: 285
['attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'attention.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'bert.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.pdf', 'gpt3.p

In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedder.encode(all_chunks, show_progress_bar=True)
embeddings = embeddings.astype(np.float32) #usually done as 64 bits but 32 is enuf

# normalize so L2 = cosine similarity
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # IP = inner product = cosine for normalized vectors, flat is to suggest that any queried vector has to be checked with every vector in the db
index.add(embeddings) #vector db created

print(f"Embeddings shape: {embeddings.shape}")
print(f"FAISS index size: {index.ntotal} vectors")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embeddings shape: (285, 384)
FAISS index size: 285 vectors


In [7]:
import pickle #so that streamlit can load the resources to its cache later


data = {
    'chunks': all_chunks,
    'sources': chunk_sources,
    'embeddings': embeddings
}

with open('chunks.pkl', 'wb') as f:
    pickle.dump(data, f)

print("done")

done


In [9]:

%%writefile app.py

import streamlit as st
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle
from google import genai
import os
GEMINI_KEY = os.environ.get('GEMINI_KEY')
print(GEMINI_KEY)

# UI

st.title("📚 Research Paper Q&A")
st.markdown("Ask anything about Transformers, BERT, GPT, RAG, LoRA, LLaMA and more.")

@st.cache_resource
def load_resources():
    with open('chunks.pkl', 'rb') as f:
        data = pickle.load(f)
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = np.array(data['embeddings']).astype(np.float32)
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings) #loading all the saved resources through a function
    return embedder, index, data['chunks'], data['sources']

embedder, index, all_chunks, chunk_sources = load_resources()
client = genai.Client(api_key=GEMINI_KEY)

def retrieve_chunks(query, top_k=5):
    query_vector = embedder.encode([query]).astype(np.float32)
    faiss.normalize_L2(query_vector)
    scores, indices = index.search(query_vector, top_k)
    results = []
    for i, idx in enumerate(indices[0]): #index 0 has returned 5 results for the given query
        results.append({
            'chunk': all_chunks[idx],
            'source': chunk_sources[idx],
            'score': float(scores[0][i])
        })
    return results

def generate_answer(query, retrieved_chunks):
    context = ""
    sources = set()
    for r in retrieved_chunks:
        context += f"\n---\nSource: {r['source']}\n{r['chunk']}\n"
        sources.add(r['source'])

    prompt = f"""You are a research assistant. Answer the question based ONLY on the context provided below.
If the context doesn't contain enough information, say "I don't have enough information in the provided papers to answer this."
Always mention which paper(s) you used to answer.

Context:
{context}

Question: {query}

Answer:"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text, list(sources)

# main UI
query = st.text_input("Ask a question:", placeholder="e.g. How does self-attention work?")

if st.button("Ask", type="primary") and query:
    with st.spinner("Retrieving relevant chunks..."):
        results = retrieve_chunks(query)

    with st.spinner("Generating answer..."):
        answer, sources = generate_answer(query, results)

    st.markdown("💡 Answer")
    st.write(answer)

    st.markdown("📄 Sources")
    for s in sources:
        st.markdown(f"- `{s}`")

Overwriting app.py


In [11]:

from pyngrok import ngrok
from google.colab import userdata
NGROK_KEY = userdata.get('NGROK_KEY')
key = userdata.get('GEMINI_KEY')


!GEMINI_KEY={key} streamlit run app.py &>/dev/null&

ngrok.set_auth_token(NGROK_KEY)
ngrok.kill()
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://wildlife-liability-dish.ngrok-free.dev" -> "http://localhost:8501"
